# Natural Language Toolkit - Bayesian

In [ ]:
import nltk
from nltk.classify import NaiveBayesClassifier
from pymongo import MongoClient
import pandas as pd

# 1. Dataset
train_data = [
    ({"love": True, "amazing": True}, "pos"),
    ({"excellent": True, "service": True}, "pos"),
    ({"great": True, "experience": True}, "pos"),
    ({"terrible": True, "awful": True}, "neg"),
    ({"worst": True, "disappointed": True}, "neg"),
    ({"bad": True, "service": True}, "neg"),
]

classifier = NaiveBayesClassifier.train(train_data)

# 2. db connection
client = MongoClient("mongodb://localhost:27017/")
db = client["sentiment_demo"]
posts_collection = db["posts"]

In [8]:
# clean collection
posts_collection.delete_many({})

DeleteResult({'n': 5, 'ok': 1.0}, acknowledged=True)

In [ ]:
# 3. post
posts = [
    {"_id": 1, "text": "I really love this new project, it's amazing!"},
    {"_id": 2, "text": "This update is terrible and breaks everything."},
    {"_id": 3, "text": "The movie was just okay, notihing special."},
    {"_id": 4, "text": "Excellent service, I will definitely come back."},
    {"_id": 5, "text": "Worst experience ever, totally disappointed"}
]

posts_collection.insert_many(posts)

InsertManyResult([1, 2, 3, 4, 5], acknowledged=True)

In [20]:
# 4. Classification
def extract_features(text):
    words = text.lower().split()
    return {w: True for w in words}

for post in posts_collection.find():
    features = extract_features(post["text"])
    sentiment = classifier.classify(features)
    posts_collection.update_one(
        {"_id": post["_id"]},
        {"$set": {"sentiment": sentiment}}
    )

    print(f"Post ID {post['_id']} classified: {sentiment}")

Post ID 1 classified: pos
Post ID 2 classified: neg
Post ID 3 classified: pos
Post ID 4 classified: pos
Post ID 5 classified: neg


In [21]:
# 5. MongoDB Query
print("Post counting")
pipeline = [
    {"$group": {"_id": "$sentiment", "count": {"$sum": 1}}}
]

for result in posts_collection.aggregate(pipeline):
    print(result)

Post counting
{'_id': 'pos', 'count': 3}
{'_id': 'neg', 'count': 2}


In [22]:
# 6. Dataframe
df = pd.DataFrame(list(posts_collection.find({}, {"_id": 1, "text":1, "sentiment":1})))
print("Post dataframe")
print(df)

Post dataframe
   _id                                             text sentiment
0    1    I really love this new project, it's amazing!       pos
1    2   This update is terrible and breaks everything.       neg
2    3       The movie was just oaky, notihing special.       pos
3    4  Excellent service, I will definitely come back.       pos
4    5      Worst experience ever, totally disappointed       neg
